# Extending Fokker-Planck Score Learning (FPSL)

Note
----
This tutorial is WIP. If your questions are not answered, please open an issue on [GitHub](https://github.com/BereauLab/fokker-planck-score-learning/issues).

In [1]:
%env XLA_PYTHON_CLIENT_PREALLOCATE = false

env: XLA_PYTHON_CLIENT_PREALLOCATE=false


# Change Noise Schedule

To change the noise schedule, you can simply create your own class which inherits from a self-defined noise schedule.

In [2]:
from dataclasses import dataclass
import jax
import jax.numpy as jnp

import fpsl
from fpsl import FPSL


# define custom noise schedule
@dataclass(kw_only=True)
class CustomNoiseSchedule(fpsl.ddm.noiseschedule.NoiseSchedule):
    """Custom noise schedule for FPSL."""

    sigma_min: float = 0.05
    sigma_max: float = 0.5

    @property
    def _noise_schedule(self) -> str:
        return 'custom'  # this name is only used for logging

    def gamma(self, t):
        # this is for legacy reasons, as FPSL used to work also on non-periodic systems
        raise NotImplementedError

    def sigma(self, t):
        # This is the default noise schedule used in FPSL.
        # return self.sigma_min ** (1 - t) * self.sigma_max**t
        return self.sigma_min ** (1 - t**2) * self.sigma_max ** (t**2)

    def beta(self, t):
        # Since we are too lazy to implement the analytical solution for the
        # custom noise schedule, we use the numerical gradient of sigma.
        return jnp.vectorize(jax.grad(lambda tt: self.sigma(tt) ** 2))(t)

Now we can define the custom FPSL by:

In [3]:
@dataclass(kw_only=True)
class CustomFPSL(CustomNoiseSchedule, FPSL):
    """Custom FPSL class with custom noise schedule."""

    pass


# generating an instance we find that is uses the new custom noise schedule
CustomFPSL(
    mlp_network=[32, 32, 32],
    key=jax.random.PRNGKey(0),
)._noise_schedule

'custom'

The same way, it is possible to change the force schedule, the prior sampling, the prior schedule, etc.

# Custom Symmetries in the Score Network

Beyond the built-in `ScorePeriodicMLP` (sine+cosine) and `ScoreSymmetricPeriodicMLP`
(cosine-only, even in every dimension) architectures, you can enforce arbitrary
symmetries by writing your own `flax.linen` score network and plugging it in via
`score_model`, exactly as we did above for the noise schedule.

As an example, suppose your data has one periodic angle `theta` and one or more
non-periodic coordinates `z` (e.g. a distance), and the underlying physics is
symmetric under simultaneously inverting both around some point:
`(z, theta) -> (2 * center - z, 2 * center - theta)`.

The trick is to build features from *products* of a basis that is odd in `z`
with a basis that is odd in `theta` (odd \* odd = even under the joint flip),
plus products of even \* even bases -- and never mix an odd factor with an
even one, since that combination flips sign under the joint reflection.

In [4]:
from functools import cached_property
from typing import Sequence

from flax import linen as nn
from fpsl.ddm.network import MLP, FourierFeatures


class CentralInversionMLP(nn.Module):
    """Score network invariant under (z, theta) -> (2*center - z, 2*center - theta)."""

    features: Sequence[int]
    fourier_features_stop: int = 4
    inversion_point: Sequence[float] = (0.5, 0.5)

    def setup(self) -> None:
        self.mlp = MLP(features=self.features)

    @nn.compact
    def __call__(self, x: jnp.ndarray, t: jnp.ndarray) -> jnp.ndarray:
        z = x[:-1] - self.inversion_point[0]
        theta = x[-1:] - self.inversion_point[1]
        K = self.fourier_features_stop

        z_odd = FourierFeatures(stop=K, odd=True, even=False)
        z_even = FourierFeatures(start=0, stop=K, odd=False, even=True)
        theta_odd = FourierFeatures(stop=K, odd=True, even=False)
        theta_even = FourierFeatures(start=0, stop=K, odd=False, even=True)

        # odd(z) * odd(theta) is even under (z, theta) -> (-z, -theta)
        odd_odd = (z_odd(z).reshape(-1, 1) * theta_odd(theta).reshape(1, -1)).reshape(
            -1
        )
        # even(z) * even(theta) is even under (z, theta) -> (-z, -theta) too
        even_even = (
            z_even(z).reshape(-1, 1) * theta_even(theta).reshape(1, -1)
        ).reshape(-1)
        # drop the constant (n=m=0) term, it carries no information
        even_even = even_even[1:]

        h = jnp.concatenate((odd_odd, even_even, t), axis=-1)
        return self.mlp(h).sum(axis=-1)

As a sanity check, this symmetry already holds for a freshly *initialized*, untrained network -- it's a structural property of the architecture, not something the network has to learn:

In [5]:
net = CentralInversionMLP(features=(16, 16), fourier_features_stop=4)
x = jnp.array([0.3, 0.8])
t = jnp.zeros((1,))
params = net.init(jax.random.PRNGKey(0), x, t)

x_reflected = (2 * jnp.array([0.5, 0.5]) - x) % 1.0
out = net.apply(params, x, t)
out_reflected = net.apply(params, x_reflected, t)
print(f'{out=}, {out_reflected=}')
assert jnp.allclose(out, out_reflected, atol=1e-5)

out=Array(-0.00264338, dtype=float32), out_reflected=Array(-0.00264336, dtype=float32)


Now plug it into `FPSL` the same way as the custom noise schedule above, by overriding the `score_model` property:

In [6]:
@dataclass(kw_only=True)
class CentralInversionFPSL(FPSL):
    """FPSL with the custom central-inversion-symmetric score network."""

    @cached_property
    def score_model(self):
        return CentralInversionMLP(
            self.mlp_network,
            fourier_features_stop=self.fourier_features,
        )


model = CentralInversionFPSL(
    mlp_network=(16, 16),
    key=jax.random.PRNGKey(0),
    n_epochs=3,
    batch_size=8,
    warmup_steps=1,
)
X = jax.random.uniform(jax.random.PRNGKey(1), (32, 2))
y = jax.random.normal(jax.random.PRNGKey(2), (32, 2))
lrs = jnp.array([1e-4, 1e-3])
loss_hist = model.train(X, y, lrs)

e1 = model.energy(x[None], t=0.0)
e2 = model.energy(x_reflected[None], t=0.0)
print(f'trained energy(x)={e1}, energy(reflected x)={e2}')
assert jnp.allclose(e1, e2, atol=1e-4)

  0%|          | 0/3 [00:00<?, ?it/s]

loss=1.062/1.062:   0%|          | 0/3 [00:00<?, ?it/s]

loss=1.062/1.062:  33%|███▎      | 1/3 [00:00<00:01,  1.41it/s]

loss=0.7921/0.7921:  33%|███▎      | 1/3 [00:00<00:01,  1.41it/s]

loss=0.9061/0.7921:  33%|███▎      | 1/3 [00:00<00:01,  1.41it/s]

loss=0.9061/0.7921: 100%|██████████| 3/3 [00:00<00:00,  4.16it/s]

trained energy(x)=[-1.6634178], energy(reflected x)=[-1.6634177]


The same product-of-bases idea generalizes to other custom symmetries: pick an
odd/even basis per coordinate (or group of coordinates), combine the pieces
whose product transforms the way you want under your symmetry, and drop
whichever leftover constant term carries no information.